In [2]:
from google.colab import drive
import os

print("Montando o Google Drive...")
drive.mount('/content/drive')

# Cria a pasta do projeto no seu Drive caso ela ainda não exista
os.makedirs('/content/drive/MyDrive/FOCA_IA/models', exist_ok=True)
print("Drive montado e pasta de destino pronta!")

Montando o Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montado e pasta de destino pronta!


In [3]:
import kagglehub
import os

# Faz o download para o cache local do Colab
caminho_dataset = kagglehub.dataset_download("lylmsc/wider-face-for-yolo-training")

print(f"Pasta raiz do dataset: {caminho_dataset}")

# Procura o arquivo data.yaml dentro da pasta baixada
caminho_yaml = os.path.join(caminho_dataset, "data.yaml")

if os.path.exists(caminho_yaml):
    print(f"data.yaml encontrado com sucesso em: {caminho_yaml}")
else:
    print("data.yaml não encontrado na raiz. Precisaremos investigar a pasta.")

100%|██████████| 2.45G/2.45G [02:21<00:00, 18.6MB/s]

Extracting files...


Pasta raiz do dataset: /root/.cache/kagglehub/datasets/lylmsc/wider-face-for-yolo-training/versions/1
data.yaml não encontrado na raiz. Precisaremos investigar a pasta.


In [5]:
import os
import shutil
import random
import yaml

print("Iniciando o particionamento do dataset")

# estrutura original do dataset (apenas treino)
caminho_origem = caminho_dataset
pasta_imgs_origem = os.path.join(caminho_origem, 'images')
pasta_lbls_origem = os.path.join(caminho_origem, 'labels')

# nova estrutura
pasta_base_foca = '/content/dataset_foca'
pastas = ['train/images', 'train/labels', 'val/images', 'val/labels']

for p in pastas:
    os.makedirs(os.path.join(pasta_base_foca, p), exist_ok=True)

# embaralha as imagens
imagens = [img for img in os.listdir(pasta_imgs_origem) if img.endswith(('.jpg', '.png'))]
random.shuffle(imagens)

# calcula a divisão 80% (treino) / 20% (validação)
corte = int(len(imagens) * 0.8)
treino_imgs = imagens[:corte]
val_imgs = imagens[corte:]

def copiar_arquivos(lista_arquivos, destino):
    for img_nome in lista_arquivos:
        # copia a imagem
        shutil.copy(os.path.join(pasta_imgs_origem, img_nome),
                    os.path.join(pasta_base_foca, destino, 'images', img_nome))

        # copia o rótulo (txt) correspondente
        lbl_nome = img_nome.replace('.jpg', '.txt').replace('.png', '.txt')
        caminho_lbl_origem = os.path.join(pasta_lbls_origem, lbl_nome)

        if os.path.exists(caminho_lbl_origem):
            shutil.copy(caminho_lbl_origem,
                        os.path.join(pasta_base_foca, destino, 'labels', lbl_nome))

print(f"Copiando {len(treino_imgs)} imagens para Treino")
copiar_arquivos(treino_imgs, 'train')

print(f"Copiando {len(val_imgs)} imagens para Validação")
copiar_arquivos(val_imgs, 'val')

# criando o arquivo data.yaml
caminho_yaml_foca = os.path.join(pasta_base_foca, 'data.yaml')
dados_yaml = {
    'path': pasta_base_foca,
    'train': 'train/images',
    'val': 'val/images',
    'nc': 1,
    'names': ['face']
}

with open(caminho_yaml_foca, 'w') as f:
    yaml.dump(dados_yaml, f)

print(f"\nDataset particionado e data.yaml criado em: {caminho_yaml_foca}")

Iniciando o particionamento do dataset
Copiando 10304 imagens para Treino
Copiando 2576 imagens para Validação

Dataset particionado e data.yaml criado em: /content/dataset_foca/data.yaml


Separamos as imagens em treino e teste

In [6]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.3 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/dataset_foca/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project='/content/drive/MyDrive/FOCA_IA/models',
    name='yolov8_nano_foca'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_foca/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0